# TP1 – Continual Learning on Seq-CIFAR-10

**I309 – Visión Artificial Avanzada – Universidad de San Andrés**

This notebook covers the full pipeline:
1. **Configuration** – all hyperparameters in one place
2. **Backbone pre-training** with Supervised Contrastive Loss (SupCon)
3. **Four CL methods**: Fine-Tune, EWC, LwF, Co²L
4. **Comparison**: unified table + accuracy and forgetting curves

> Run cells top-to-bottom. Skip *Section 2* if `backbone.pth` already exists.  
> Skip *Section 3* if `graphs/results.json` already exists (use `--skip-training` logic in the load cell).

---
## 0 · Imports & device

In [ ]:
import os, json, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# ── project modules ───────────────────────────────────────────────────────────
from data.prepare_cifar10 import task_splits, TASKS, CIFAR10_CLASSES
from models.train_backbone import (
    BackboneModel, build_backbone, train_backbone,
    EMBEDDING_DIM, TEMPERATURE,
)
from models.finetune import train_finetune
from models.ewc      import train_ewc
from models.lwf      import train_lwf
from models.co2l     import train_co2l

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## 1 · Configuration

Change any value here — the rest of the notebook picks it up automatically.

In [ ]:
# ── Backbone pre-training ─────────────────────────────────────────────────────
BACKBONE_EPOCHS  = 25       # epochs for SupCon pre-training (task 0 only)
BACKBONE_LR      = 5e-3
BACKBONE_TEMP    = 0.07     # SupCon temperature τ
BACKBONE_WEIGHTS = 'backbone.pth'

# ── Common CL hyperparameters ─────────────────────────────────────────────────
CL_EPOCHS     = 5           # backbone/joint-head training epochs per task
CL_LR         = 1e-3
BATCH_SIZE    = 64
HEAD_EPOCHS   = 5           # Task-IL head training epochs (frozen backbone)
HEAD_LR       = 8e-4

# ── Method-specific ───────────────────────────────────────────────────────────
LAMBDA_EWC    = 100.0       # EWC regularisation strength  (try 1 000–10 000)
FISHER_SAMPLES = 200        # samples used to estimate Fisher diagonal

LAMBDA_LWF    = 1.0         # LwF distillation weight
TEMPERATURE_LWF = 2.0       # soft-target temperature

LAMBDA_CO2L   = 1.0         # Co²L distillation weight
BUFFER_SIZE   = 200         # replay buffer capacity (total across all tasks)

# ── Output ────────────────────────────────────────────────────────────────────
GRAPHS_DIR   = 'graphs'
RESULTS_PATH = os.path.join(GRAPHS_DIR, 'results.json')
os.makedirs(GRAPHS_DIR, exist_ok=True)

# ── Which methods to run ──────────────────────────────────────────────────────
# Set to False to skip a method (results still loaded from cache if available)
RUN_FINETUNE = True
RUN_EWC      = True
RUN_LWF      = True
RUN_CO2L     = True

print('Configuration loaded ✓')

---
## 1.5 · Dataset overview

In [ ]:
print('Seq-CIFAR-10 split:')
print(f'{"Task":<6} {"Classes":<30} {"Train":>7} {"Test":>7}')
print('-' * 55)
for i, (split, cls) in enumerate(zip(task_splits, TASKS)):
    names = ', '.join(CIFAR10_CLASSES[c] for c in cls)
    print(f'{i:<6} {names:<30} {len(split["train"]):>7} {len(split["test"]):>7}')

---
## 2 · Backbone pre-training (Etapa 4.2)

Trains a ResNet-18 backbone on **Task 0** with Supervised Contrastive Loss and saves `backbone.pth`.  
**Skip this cell if `backbone.pth` already exists.**

In [ ]:
if os.path.exists(BACKBONE_WEIGHTS):
    print(f'Found {BACKBONE_WEIGHTS} — skipping training.')
else:
    print(f'Training backbone for {BACKBONE_EPOCHS} epochs on Task 0 …')
    train_backbone(
        task_number=0,
        num_epochs=BACKBONE_EPOCHS,
        lr=BACKBONE_LR,
        temperature=BACKBONE_TEMP,
        batch_size=BATCH_SIZE,
        save_path=BACKBONE_WEIGHTS,
    )
    print('Backbone saved ✓')

### 2.1 · t-SNE embedding visualisation

In [ ]:
snap_path = os.path.join(GRAPHS_DIR, 'latent_space_snapshots.png')
if os.path.exists(snap_path):
    from IPython.display import Image, display
    display(Image(snap_path))
else:
    print('Run backbone training first to generate snapshots.')

---
## 3 · Continual Learning methods (Etapa 4.3)

Each method trains sequentially over all 5 tasks and records Class-IL and Task-IL accuracy after each task.  
Results are cached in `graphs/results.json` — re-run only the methods you change.

In [ ]:
# Load existing results (if any) so we don't re-run completed methods
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        results = json.load(f)
    print(f'Loaded cached results for: {list(results.keys())}')
else:
    results = {}
    print('No cached results — will run all selected methods.')

def _save_results():
    with open(RESULTS_PATH, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'Saved → {RESULTS_PATH}')

### 3.1 · Fine-Tune (naive baseline)

In [ ]:
if RUN_FINETUNE:
    print('Running Fine-Tune …')
    results['Fine-Tune'] = train_finetune(
        backbone_weights=BACKBONE_WEIGHTS,
        num_epochs=CL_EPOCHS, lr=CL_LR, batch_size=BATCH_SIZE,
        head_epochs=HEAD_EPOCHS, head_lr=HEAD_LR,
    )
    _save_results()
else:
    print('Fine-Tune skipped (RUN_FINETUNE=False)')

### 3.2 · Elastic Weight Consolidation (EWC)

In [ ]:
if RUN_EWC:
    print('Running EWC …')
    results['EWC'] = train_ewc(
        backbone_weights=BACKBONE_WEIGHTS,
        lambda_ewc=LAMBDA_EWC, fisher_samples=FISHER_SAMPLES,
        num_epochs=CL_EPOCHS, lr=CL_LR, batch_size=BATCH_SIZE,
        head_epochs=HEAD_EPOCHS, head_lr=HEAD_LR,
    )
    _save_results()
else:
    print('EWC skipped (RUN_EWC=False)')

### 3.3 · Learning without Forgetting (LwF)

In [ ]:
if RUN_LWF:
    print('Running LwF …')
    results['LwF'] = train_lwf(
        backbone_weights=BACKBONE_WEIGHTS,
        lambda_lwf=LAMBDA_LWF, temperature=TEMPERATURE_LWF,
        num_epochs=CL_EPOCHS, lr=CL_LR, batch_size=BATCH_SIZE,
        head_epochs=HEAD_EPOCHS, head_lr=HEAD_LR,
    )
    _save_results()
else:
    print('LwF skipped (RUN_LWF=False)')

### 3.4 · Contrastive Continual Learning (Co²L)

In [ ]:
if RUN_CO2L:
    print('Running Co²L …')
    results['Co2L'] = train_co2l(
        backbone_weights=BACKBONE_WEIGHTS,
        lambda_co2l=LAMBDA_CO2L, temperature=TEMPERATURE,
        buffer_size=BUFFER_SIZE,
        num_epochs=CL_EPOCHS, lr=CL_LR, batch_size=BATCH_SIZE,
        head_epochs=HEAD_EPOCHS, head_lr=HEAD_LR,
    )
    _save_results()
else:
    print('Co²L skipped (RUN_CO2L=False)')

---
## 4 · Comparison of results (Etapa 4.4)

### 4.1 · Unified results table

In [ ]:
METHODS = ['Fine-Tune', 'EWC', 'LwF', 'Co2L']
COLORS  = ['tab:red', 'tab:orange', 'tab:blue', 'tab:green']
TASK_LABELS = [f'T{i}' for i in range(1, 6)]

col_w = 12
header = f'{"Method":<{col_w}}' + ''.join(
    f'  {"T"+str(t)+" CIL":>8}  {"T"+str(t)+" TIL":>8}' for t in range(1, 6)
)
sep = '=' * len(header)
print(sep)
print('RESULTS — Class-IL (CIL) and Task-IL (TIL) accuracy (%) after each task')
print(sep)
print(header)
print('-' * len(header))
for method in METHODS:
    if method not in results:
        continue
    r = results[method]
    row = f'{method:<{col_w}}'
    for i in range(5):
        row += f'  {r["class_il"][i]:>8.2f}  {r["task_il"][i]:>8.2f}'
    print(row)
print(sep)

### 4.2 · Accuracy curves

In [ ]:
tasks = list(range(1, 6))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Continual Learning on Seq-CIFAR-10', fontsize=14, fontweight='bold')

for method, color in zip(METHODS, COLORS):
    if method not in results:
        continue
    r = results[method]
    ax1.plot(tasks, r['class_il'], marker='o', label=method, color=color, linewidth=2)
    ax2.plot(tasks, r['task_il'],  marker='o', label=method, color=color, linewidth=2)

for ax, title in [
    (ax1, 'Class-Incremental Learning (Class-IL)'),
    (ax2, 'Task-Incremental Learning (Task-IL)'),
]:
    ax.set_xlabel('Tasks Learned', fontsize=11)
    ax.set_ylabel('Accuracy (%)', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.set_xticks(tasks)
    ax.set_xticklabels(TASK_LABELS)
    ax.set_ylim(0, 100)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
path = os.path.join(GRAPHS_DIR, 'accuracy_curves.png')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {path}')

### 4.3 · Forgetting curves

Class-IL accuracy over tasks acts as a forgetting proxy: a steeper drop means more catastrophic forgetting.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.set_title(
    'Forgetting — Class-IL Accuracy Over Tasks\n'
    '(steeper drop = more catastrophic forgetting)',
    fontsize=12,
)
for method, color in zip(METHODS, COLORS):
    if method not in results:
        continue
    r = results[method]
    ax.plot(tasks, r['class_il'], marker='o', label=method, color=color, linewidth=2)

ax.set_xlabel('Tasks Learned', fontsize=11)
ax.set_ylabel('Class-IL Accuracy (%)', fontsize=11)
ax.set_xticks(tasks)
ax.set_xticklabels(TASK_LABELS)
ax.set_ylim(0, 100)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
path = os.path.join(GRAPHS_DIR, 'forgetting_curves.png')
plt.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {path}')

### 4.4 · Final accuracy summary (after all 5 tasks)

In [ ]:
print(f'{"Method":<12}  {"Final Class-IL":>15}  {"Final Task-IL":>14}')
print('-' * 46)
for method in METHODS:
    if method not in results:
        continue
    r = results[method]
    print(f'{method:<12}  {r["class_il"][-1]:>14.2f}%  {r["task_il"][-1]:>13.2f}%')

print('\nPaper targets (Co²L on Seq-CIFAR-10):')
print('  Class-IL ≈ 47–52%   Task-IL ≈ 88–92%')